# 第 2 章 数字傅里叶变换 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/waveprop/ch2/`。

## case · studies

`ch2_case_studies.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 2 章 =====
三个变换案例研究：sinc / 高斯 / 带二次相位的高斯（对应原书 2.5 节）

三种信号的带宽类型不同，采样间隔 delta 的确定方法也不同：
    1) sinc(a x)：硬限带，G = (1/a) rect(f/a)，带宽 |f| <= a/2，
       奈奎斯特判据 delta <= 1/a（恰好两倍余量时取 delta = 0.85/a，原书图 2.10）；
    2) exp(-pi (a x)^2)：软限带，G = (1/|a|) exp(-pi (f/a)^2)。
       按 1/e^2 频率取 delta_e2 = (1/(2a)) sqrt(pi/2)（原书式 2.25），
       频带边缘可见混叠；改按 p=0.01 电平取 delta（原书式 2.26/2.27）后混叠消失；
    3) exp(-pi (a x)^2) * exp(i pi (b x)^2)：二次相位把带宽拉宽，
       p 电平带宽由原书式 (2.31)：f_p = sqrt(-(a^2 + b^4/a^2)/pi * ln p)。
原书勘误（本脚本数值复现）：
    原书 2.5.3 节称 a=0.25、b=0.57 时 "f_{x,0.01} = 1.96 m^-1"，但按式 (2.31)
    代入 p=0.01 得 1.60 m^-1；1.96 m^-1 实为 p=0.001 的结果。
数值验证（运行自动打印）：
    - 案例 1：幅度波纹、频带边缘的相位 pi 跳变计数；
    - 案例 2：delta_e2 与 delta_0.01 的最左样本混叠抬升对比；
    - 案例 3：式 (2.31) 的 p 电平带宽 vs 数值谱、原书两个数值的归属，
      以及式 (2.29) 解析 FT 与直接积分的核对、图 2.14 三条 b 曲线的带宽。
MATLAB 注意：numpy 的 np.sinc(x) 是 sin(pi x)/(pi x)（归一化），
与 MATLAB Signal Processing Toolbox 的 sinc 同定义；为不添依赖，
本脚本自带 mysinc（与 base.py 的 sinc 相同，此处从 base 导入）。
用法：python ch2_case_studies.py  （纯数值输出，无图形）
依赖：numpy；base.py（同仓库 python/fourier/common）。
"""
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import sinc, coords  # noqa: E402


def ft1(g, delta):
    """居中一维 DFT（原书 Listing 2.1）"""
    return np.fft.fftshift(np.fft.fft(np.fft.fftshift(g))) * delta


def ft_ana_sinc(f, a):
    """sinc(a x) 的解析 FT：(1/a) rect(f/a)（原书式 2.22）"""
    return np.where(np.abs(f) < a / 2, 1.0 / a,
                    np.where(np.abs(f) == a / 2, 0.5 / a, 0.0))


def ft_ana_gauss(f, a):
    """exp(-pi (a x)^2) 的解析 FT（原书式 2.24）"""
    return 1.0 / abs(a) * np.exp(-np.pi * (f / a) ** 2)


def ft_ana_gauss_quad(f, a, b):
    """带二次相位高斯的解析 FT（原书式 2.29）"""
    c = a ** 2 - 1j * b ** 2
    return 1.0 / np.sqrt(c) * np.exp(-np.pi * f ** 2 / c)


def bandwidth_p(a, b, p):
    """p 电平带宽（原书式 2.31；b=0 时退化为式 2.27）"""
    return np.sqrt(-(a ** 2 + b ** 4 / a ** 2) / np.pi * np.log(p))


def main():
    print('===== 案例 1：sinc 信号（原书图 2.10：a=1.1, delta=0.85/a, N=32）=====')
    a = 1.1
    delta = 0.85 / a
    N = 32
    x, f = coords(N * delta, N)
    g = sinc(a * x)
    G = ft1(g, delta)
    G_a = ft_ana_sinc(f, a)
    band = np.abs(f) <= a / 2
    rip = np.max(np.abs(np.abs(G) - np.abs(G_a))[band]) * a
    edge = ~band
    ph_edge = np.angle(G[edge])
    frac_pi = np.mean(np.abs(np.abs(ph_edge) - np.pi) < 0.1)
    print('    带内幅度波纹 = %.2e（相对峰值 1/a=%.2f）' % (rip, 1 / a))
    print('    带外样本中相位≈pi 的比例 = %.0f%%（谱值应为 0，实为小负数）'
          % (100 * frac_pi))

    print('===== 案例 2：高斯信号（原书图 2.11/2.12：a=1）=====')
    a = 1.0
    f_e2 = a * np.sqrt(2 / np.pi)
    delta_e2 = 1.0 / (2 * f_e2)
    delta_01 = 1.0 / (2 * bandwidth_p(a, 0.0, 0.01))
    for tag, d in [('delta_e2  = %.4f' % delta_e2, delta_e2),
                   ('delta_0.01 = %.4f' % delta_01, delta_01)]:
        N = 32
        x, f = coords(N * d, N)
        G = ft1(np.exp(-np.pi * a * x ** 2), d)
        i_left = 0                      # 居中排列的最左样本 = 频带负边缘
        lift = G[i_left].real - ft_ana_gauss(f[i_left], a)
        print('    %s：最左样本 f=%.3f，混叠抬升 = %.4f（解析值 %.4f，抬升 %.0f%%）'
              % (tag, f[i_left], G[i_left].real,
                 ft_ana_gauss(f[i_left], a),
                 100 * lift / ft_ana_gauss(0, a)))

    print('===== 案例 3：二次相位高斯（原书图 2.13–2.15）=====')
    a, b = 0.25, 0.57
    # --- 式 (2.31) 的 p 电平带宽 vs 数值谱的 p 电平位置 ---
    for p in (0.01, 0.001):
        fp = bandwidth_p(a, b, p)
        # 数值：细网格上解析谱幅值降到 p 倍峰值的频率（验证式 2.29/2.31）
        f_fine = np.linspace(0, 4, 400001)
        A = np.abs(ft_ana_gauss_quad(f_fine, a, b))
        fp_num = np.interp(p, (A / A[0])[::-1], f_fine[::-1])
        print('    p=%-6g：式(2.31) f_p = %.4f m^-1，数值谱电平位置 = %.4f m^-1'
              % (p, fp, fp_num))
    print('    -> 原书正文"f_{x,0.01} = 1.96 m^-1"实为 p=0.001 的值（勘误）；'
          'p=0.01 应为 %.2f m^-1' % bandwidth_p(a, b, 0.01))
    # --- 式 (2.29) 解析 FT vs 直接数值积分 ---
    f0 = 0.9
    xx = np.linspace(-40, 40, 1_200_001)
    g_num = np.exp(-np.pi * (a * xx) ** 2) * np.exp(1j * np.pi * (b * xx) ** 2)
    G_num = np.trapezoid(g_num * np.exp(-2j * np.pi * f0 * xx), xx)
    rel = abs(G_num - ft_ana_gauss_quad(f0, a, b)) / abs(ft_ana_gauss_quad(f0, a, b))
    print('    式(2.29) 在 f=0.9 处 vs 直接积分：相对偏差 = %.2e' % rel)
    # --- 原书采样参数下的 DFT 精度 ---
    for tag, d, N in [('原书 delta=0.25 m（=p=0.001 判据）', 0.25, 40),
                      ('p=0.01 判据 delta=%.3f m' % (1 / (2 * bandwidth_p(a, b, 0.01))),
                       1 / (2 * bandwidth_p(a, b, 0.01)), 32)]:
        x, f = coords(N * d, N)
        g = np.exp(-np.pi * (a * x) ** 2) * np.exp(1j * np.pi * (b * x) ** 2)
        G = ft1(g, d)
        G_a = ft_ana_gauss_quad(f, a, b)
        ok = np.abs(G_a) > 1e-3 * np.abs(G_a).max()
        amp_err = np.max(np.abs(np.abs(G) - np.abs(G_a))[ok]) / np.abs(G_a).max()
        ph_err = np.max(np.abs(np.angle(G / G_a))[ok])
        print('    %s N=%d：带内幅度误差 %.2e，相位误差 %.2e rad'
              % (tag, N, amp_err, ph_err))
    # --- 原书图 2.14：a=0.33，b 增大拉宽带宽 ---
    print('    图 2.14 的带宽（a=0.33，p=0.01）：')
    for bb in (0.75, 1.5, 2.5):
        print('      b=%-4.2f -> f_p = %6.2f m^-1' % (bb, bandwidth_p(0.33, bb, 0.01)))


if __name__ == '__main__':
    main()


## discretization

`ch2_discretization.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 2 章 =====
离散化的三种失真与三条补救路线（对应原书 2.3、2.4 节与图 2.6–2.9）

连续 FT 对 g(x)=exp(-a|x|) ⇔ G(f)=2a/(a^2+(2*pi*f)^2)（原书式 (2.13)(2.14)，
a=10 m^-1，与原书图 2.6 参数一致）。DFT 链条的三步失真：
    1) 空域采样（乘 delta^-1 comb(x/delta)）  -> 频域周期复制 comb(delta*f)：混叠
    2) 空域截断（乘 rect(x/L)）              -> 频域卷积 L sinc(L f)：波纹+展宽
    3) 频域采样（乘 comb(L f)）              -> 空域虚拟周期延拓（周期 L）
其中第 1+2 步合并后的"连续频谱"有一个无需数值卷积的精确算法：
    S(f) = sum_{n=-N/2}^{N/2-1} g(n*delta) exp(-i 2 pi f n delta)
（这就是"采样 + 截断后的空域信号"的真实连续 FT；对它再在 f_m = m/(N*delta)
 上取样、乘 delta，就得到 DFT 输出——脚本内部对此做了自检。）

三条补救路线（原书图 2.7 / 2.8 / 2.9）：
    A 加大网格：delta 翻倍、N 不变     -> L=N*delta 翻倍，压波纹，但混叠变重
    B 加密采样：delta 减半、N 翻倍     -> L 不变，压混叠，但波纹不变
    C 双管齐下：delta 减半、L 也翻倍   -> N=4 倍，混叠与波纹同时下降
数值验证（运行自动打印）：
    - 混叠折返恒等式：6 m^-1 的余弦按 delta=1/8 m 采样与 2 m^-1 逐点相同；
    - delta*S(f_m) 与 ft(g) 的 DFT 输出逐 bin 一致（链条自洽）；
    - 四种策略的混叠/波纹/总误差对照表。
MATLAB 注意：本脚本无 MATLAB 专有函数；fftshift 语义与 MATLAB 相同
（偶数 N 时 fftshift == ifftshift）。
用法：python ch2_discretization.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False


def g_func(x, a):
    """空域信号 exp(-a|x|)（原书式 2.13）"""
    return np.exp(-a * np.abs(x))


def G_func(f, a):
    """解析 FT：2a/(a^2+(2 pi f)^2)（原书式 2.14，峰值 2/a）"""
    return 2 * a / (a ** 2 + (2 * np.pi * f) ** 2)


def alias_only(f, a, delta, K=400):
    """只做第 1 步的频谱：G ⊗ comb(delta f) = (1/delta) Σ_k G(f + k/delta)。
    （原书式 2.15 的空域采样对 (1/delta)comb(x/delta) 在频域产生
    间隔 1/delta、幅度放大 1/delta 的周期复制；洛伦兹尾按 1/f^2 衰减，
    K=400 个周期后截断误差可忽略。）"""
    s = np.zeros_like(f, dtype=float)
    for k in range(-K, K + 1):
        s += G_func(f + k / delta, a)
    return s / delta


def chain_spectrum(f, g_samp, delta):
    """第 1+2 步的精确频谱 S(f) = Σ g(n delta) exp(-i2 pi f n delta)。
    g_samp 按居中排列（n = -N/2 .. N/2-1），返回复数谱（长度同 f）。"""
    n = np.arange(len(g_samp)) - len(g_samp) // 2
    terms = g_samp[np.newaxis, :] * np.exp(
        -2j * np.pi * f[:, np.newaxis] * (n * delta)[np.newaxis, :])
    return terms.sum(axis=1)


def ft1(g, delta):
    """居中一维 DFT（原书 Listing 2.1；等价于 base.py 的 ft1_centered）"""
    return np.fft.fftshift(np.fft.fft(np.fft.fftshift(g))) * delta


def main(outdir=None):
    a = 10.0
    # ---- 四种策略：(名称, delta, L, N) ----
    strategies = [
        ('基线（原书图 2.6）',        0.0375,  0.6, 16),
        ('A 加大网格：delta x2（图 2.7）', 0.075,  1.2, 16),
        ('B 加密采样：N x2（图 2.8）',   0.01875, 0.6, 32),
        ('C 双管齐下（图 2.9）',        0.01875, 1.2, 64),
    ]

    print('===== DFT 链条自检：delta*S(f_m) 与 DFT 输出逐 bin 对比 =====')
    f_fine = np.linspace(-20.0, 20.0, 8001)
    for name, delta, L, N in strategies:
        x = (np.arange(N) - N // 2) * delta
        g_samp = g_func(x, a)
        S = chain_spectrum(f_fine, g_samp, delta)
        fm = (np.arange(N) - N // 2) / (N * delta)      # DFT bin 频率
        S_bins = chain_spectrum(fm, g_samp, delta)
        G_dft = ft1(g_samp, delta)
        err = np.max(np.abs(delta * S_bins - G_dft))
        print('  %-24s max|delta*S - DFT| = %.2e' % (name, err))

    print('\n===== 混叠折返恒等式（原书图 2.5 第一组：f1=6, delta=1/8 m）=====')
    n_ali = np.arange(-16, 16)
    x_ali = n_ali / 8.0
    d_max = np.max(np.abs(np.cos(2 * np.pi * 6 * x_ali)
                          - np.cos(2 * np.pi * 2 * x_ali)))
    print('    max|cos(2*pi*6*x) 与 cos(2*pi*2*x) 的样本差| = %.1e（f_s=8, 折返 f2=2）'
          % d_max)

    print('\n===== 三种失真的量化（频带 |f| <= 1/(2 delta)，误差以解析峰值为基准）=====')
    print('  %-24s %-14s %-14s %-14s' %
          ('策略', '混叠误差', '波纹误差', 'DFT 总误差'))
    results = []
    for name, delta, L, N in strategies:
        x = (np.arange(N) - N // 2) * delta
        g_samp = g_func(x, a)
        fmax = 1.0 / (2 * delta)
        band = np.abs(f_fine) <= fmax
        G_true = G_func(f_fine, a)
        G_ali = alias_only(f_fine, a, delta)     # (1/delta)ΣG，放大标度
        S = chain_spectrum(f_fine, g_samp, delta).real   # 同为放大标度
        e_alias = np.max(np.abs(delta * G_ali - G_true)[band]) / G_func(0, a)
        e_rip = np.max(np.abs(delta * S - delta * G_ali)[band]) / G_func(0, a)
        fm = (np.arange(N) - N // 2) / (N * delta)
        e_dft = np.max(np.abs(ft1(g_samp, delta)
                              - G_func(fm, a))) / G_func(0, a)
        results.append((name, delta, L, N, f_fine, G_true, G_ali, S,
                        fm, ft1(g_samp, delta), e_alias, e_rip, e_dft))
        print('  %-24s %-14.2e %-14.2e %-14.2e'
              % (name, e_alias, e_rip, e_dft))

    # ---- 绘图：4 行 x 2 列（左列用原书图 (d)(f) 的 1/delta 放大标度）----
    fig, axes = plt.subplots(4, 2, figsize=(11.5, 13.5))
    for r, res in enumerate(results):
        name, delta, L, N, ff, Gt, Ga, S, fm, Gdft, *_ = res
        ax = axes[r, 0]
        ax.plot(ff, Gt / delta, color='0.65', lw=1.6, ls='--',
                label=r'解析 $G(f_x)/\delta$')
        ax.plot(ff, Ga, color='tab:blue', lw=1.0, ls=':',
                label=r'$G\otimes\mathrm{comb}(\delta f_x)$（混叠）')
        ax.plot(ff, S, color='k', lw=1.3,
                label=r'再 $\otimes L\,\mathrm{sinc}(Lf_x)$（+波纹）')
        ax.set_xlim(-20, 20)
        ax.set_ylim(-0.05, Ga.max() * 1.12)
        ax.set_title('%s   $\\delta$=%.4g m, L=%.2g m, N=%d'
                     % (name, delta, L, N), fontsize=10)
        ax.set_ylabel('幅值（$1/\\delta$ 放大标度）')
        if r == 0:
            ax.legend(fontsize=8, loc='upper right')
        if r == 3:
            ax.set_xlabel(r'$f_x$ [m$^{-1}$]')

        ax = axes[r, 1]
        markerline, stemlines, _ = ax.stem(fm, np.abs(Gdft))
        plt.setp(stemlines, lw=0.8)
        plt.setp(markerline, ms=3.5)
        ax.plot(ff, Gt, color='0.65', lw=1.6, ls='--', label='解析')
        ax.set_xlim(-20, 20)
        ax.set_ylim(-0.05, 0.32)
        ax.set_title('DFT 输出 vs 解析（总误差 %.1e）' % res[12], fontsize=10)
        if r == 0:
            ax.legend(fontsize=8, loc='upper right')
        if r == 3:
            ax.set_xlabel(r'$f_x$ [m$^{-1}$]')

    for ax in axes.ravel():
        ax.grid(alpha=0.25)
    fig.tight_layout()

    if outdir:
        os.makedirs(outdir, exist_ok=True)
        fp = os.path.join(outdir, 'dft_remedies.png')
        fig.savefig(fp, dpi=150)
        print('\n已保存：%s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## ft2 · 2d

`ch2_ft2_2d.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 2 章 =====
二维 DFT：ft2 / ift2 与方形孔径的频谱（对应原书 2.6 节 Listings 2.5–2.6）

复用说明：
    base.py（python/fourier/common/）的 ft2_centered 与本脚本按原书定义的
    ft2 在偶数 N 下完全一致；sinc 也取自 base（归一化定义 sin(pi x)/(pi x)）。
    import 路径处理同 ch2_ft_basics.py。

示例：方形孔径 g(x,y) = rect(x/D) rect(y/D)，其连续 FT 为
    G(fx,fy) = D^2 sinc(D fx) sinc(D fy)
这正是第 4 章以后"夫琅禾费衍射图样 = 孔径 FT"的最小例子（方孔的远场
衍射图样）。相位在 sinc 过零点两侧翻转 0/pi，呈棋盘结构。

数值验证（运行自动打印）：
    (1) ift2(ft2(g)) 往返最大误差；
    (2) 二维 Parseval：sum|g|^2 dx^2 = sum|G|^2 dfx dfy；
    (3) 中心样本 G(0,0) = 孔径面积 D^2；
    (4) fy=0 剖面 vs 解析 D^2 sinc(D fx) 的最大相对偏差；
    (5) 原书 Listing 2.6 的 ift2 假定方阵（N = size(G,1) 同时用于两维），
        本脚本对非方阵验证"必须用 (M*dfx)*(N*dfy)"的修正版。
MATLAB/Python 差异：
    - MATLAB ift2 用 (N*delta_f)^2 且 N 取 size(G,1)——仅对方阵正确；
      numpy 侧本脚本写成对 M x N 通用的形式。
    - imshow 默认 origin='upper'（图像惯例），MATLAB imagesc + axis xy
      才与数学坐标一致；本脚本显式传 origin='lower'。
用法：python ch2_ft2_2d.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib；base.py（同仓库 python/fourier/common）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import sinc  # noqa: E402


# ===== 书 2.6 节 Listing 2.5 / 2.6：二维定标正确的 DFT / DIFT =====
def ft2(g, delta):
    """二维 DFT（原书 Listing 2.5）：G = fftshift(fft2(fftshift(g)))*delta^2"""
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(g))) * delta ** 2


def ift2(G, delta_f, delta_f_y=None):
    """二维 DIFT（原书 Listing 2.6 的通用化）：
    g = ifftshift(ifft2(ifftshift(G))) * (M*df_y)*(N*df_x)。
    delta_f 是 x 方向频率间隔 1/(N*delta_x)；delta_f_y 是 y 方向的
    1/(M*delta_y)，方阵且两维等间距时与 delta_f 相同（默认）。
    原书按方阵写死 (N*delta_f)^2（MATLAB 的 N=size(G,1) 是行数），
    非方阵时行/列两维的 L 各不相同，必须分开乘。"""
    M, N = G.shape
    dfy = delta_f_y if delta_f_y is not None else delta_f
    return np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(G))) \
        * (M * dfy) * (N * delta_f)


def main(outdir=None):
    # ---- 网格与孔径 ----
    N = 128            # 每边采样点数（原书约定偶数 N；FFT 对 2 的幂最快）
    L = 4.0            # 空域边长 [m] -> delta = L/N = 31.25 mm
    D = 0.5            # 方形孔径边长 [m]（D/delta = 16，恰为整数个胞元）
    delta = L / N
    delta_f = 1.0 / (N * delta)
    n = np.arange(N) - N // 2
    # 胞元中心采样（第 1 章笔记的约定）：x = (n + 1/2)*delta，孔径边界
    # 恰好落在胞元边界上 -> 离散孔径与连续 rect 逐点相等，面积严格 = D^2。
    # 若用原书 Listing 2.3 的节点采样 x = n*delta，边界样本 |x| = D/2 会被
    # 整格计入（17 格），G(0,0) 多出 2*D*delta 量级的偏差。
    x = (n + 0.5) * delta
    f = n / (N * delta)
    X, Y = np.meshgrid(x, x)          # indexing='xy'：与 MATLAB meshgrid 同义
    FX, FY = np.meshgrid(f, f)

    g = ((np.abs(X) <= D / 2) & (np.abs(Y) <= D / 2)).astype(float)
    G = ft2(g, delta)

    # ---- 数值验证 ----
    print('===== 二维 DFT 验证（方孔 rect(x/D)rect(y/D)，D=%.2f m）=====' % D)
    g_rt = ift2(G, delta_f)
    print('(1) max|ift2(ft2(g)) - g| = %.2e' % np.max(np.abs(g_rt - g)))
    lhs = np.sum(np.abs(g) ** 2) * delta ** 2
    rhs = np.sum(np.abs(G) ** 2) * delta_f ** 2
    print('(2) Parseval: sum|g|^2*dx^2 = %.10f, sum|G|^2*df^2 = %.10f (偏差 %.2e)'
          % (lhs, rhs, abs(lhs - rhs) / lhs))
    i0 = N // 2
    print('(3) |G(0,0)| = %.6f，孔径面积 D^2 = %.6f（偏差 %.2e）'
          % (abs(G[i0, i0]), D ** 2, abs(abs(G[i0, i0]) - D ** 2)))
    prof = np.abs(G[i0, :])                      # fy = 0 剖面（取幅值：胞元中心
    prof_ana = np.abs(D ** 2 * sinc(D * f))      # 采样给谱附加 pi*f*delta 线性相位）
    ok = np.abs(f) <= 2 / D                      # 主瓣 + 前两个旁瓣内比较
    rel = np.max(np.abs(prof - prof_ana)[ok]) / D ** 2
    print('(4) fy=0 剖面（|fx| <= 2/D）vs 解析 D^2 sinc(D fx)：最大偏差 %.2e'
          '（相对峰值）' % rel)

    # 非方阵检查：M x N 矩形阵列（两维 delta 相同、频率间隔不同）上，
    # 原书 ift2 的 (size(G,1)*df)^2 定标错；正确做法是两维分别乘 M*df_y、N*df_x
    M2 = 64
    gx = np.exp(-np.pi * (x / 1.0) ** 2)         # x 向（N 点）
    gy = np.exp(-np.pi * (x[:M2] / 1.0) ** 2)    # y 向（M2 点）
    g2 = np.outer(gy, gx)                        # (M2, N) 各向异性高斯
    G2 = ft2(g2, delta)
    dfx = 1.0 / (N * delta)                      # x 向频率间隔
    dfy = 1.0 / (M2 * delta)                     # y 向频率间隔
    rt_book = np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(G2))) \
        * (G2.shape[0] * delta_f) ** 2           # 原书写法：MATLAB size(G,1)=行数 M2
    rt_fix = ift2(G2, dfx, delta_f_y=dfy)
    print('(5) %dx%d 非方阵：原书 (行数*df_x)^2 往返误差 %.2e；'
          ' (M*df_y)(N*df_x) 往返误差 %.2e'
          % (M2, N, np.max(np.abs(rt_book - g2)), np.max(np.abs(rt_fix - g2))))

    # ---- 绘图 ----
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
    im0 = axes[0].imshow(g, extent=[x[0], x[-1], x[0], x[-1]],
                         origin='lower', cmap='gray')
    axes[0].set_title('孔径 $g(x,y)=\\mathrm{rect}(x/D)\\mathrm{rect}(y/D)$')
    axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('y [m]')
    fig.colorbar(im0, ax=axes[0], shrink=0.85)

    im1 = axes[1].imshow(20 * np.log10(np.abs(G) / np.abs(G).max() + 1e-12),
                         extent=[f[0], f[-1], f[0], f[-1]],
                         origin='lower', cmap='magma', vmin=-60, vmax=0)
    axes[1].set_title(r'$|G(f_x,f_y)|$ [dB]，$G=D^2\,\mathrm{sinc}\,\mathrm{sinc}$')
    axes[1].set_xlabel(r'$f_x$ [m$^{-1}$]'); axes[1].set_ylabel(r'$f_y$ [m$^{-1}$]')
    fig.colorbar(im1, ax=axes[1], shrink=0.85)

    axes[2].plot(f[ok], prof_ana[ok], color='0.6', lw=2, ls='--',
                 label=r'解析 $D^2\,\mathrm{sinc}(Df_x)$')
    axes[2].plot(f[ok], prof[ok], 'b.', ms=4, label='DFT 剖面 ($f_y$=0)')
    axes[2].set_xlabel(r'$f_x$ [m$^{-1}$]'); axes[2].set_ylabel('幅值')
    axes[2].set_title('剖面对比（最大偏差 %.1e）' % rel)
    axes[2].legend(fontsize=9)
    for ax in axes:
        ax.grid(alpha=0.2)
    fig.tight_layout()

    if outdir:
        os.makedirs(outdir, exist_ok=True)
        fp = os.path.join(outdir, 'ft2_demo.png')
        fig.savefig(fp, dpi=150)
        print('已保存：%s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## ft · 基础函数

`ch2_ft_basics.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 2 章 =====
DFT 基础：定标、重排、正逆变换配对（对应原书 2.1 节与 Listings 2.1–2.4）

复用说明：
    本脚本从 python/fourier/common/base.py 导入公共函数（sinc/gauss/coords）。
    base.py 属《计算傅里叶光学》(Voelz) 笔记的公共模块，其中的
    ft1_centered/ft2_centered 与本节按原书定义的 ft/ft2 在偶数 N 下完全等价
    （偶数 N 时 fftshift 与 ifftshift 相同）。import 路径处理：

        import os, sys
        sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                        "..", "..", "fourier", "common"))
        from base import gauss, coords

数值验证（运行本脚本全部自动打印）：
    (1) 原书式 (2.6) 的 DFT 与"逐点黎曼和"一致（证明 fft 确实算的是那个和）；
    (2) 原书式 (2.5) 的重排公式按字面无法实现（原书笔误），修正公式与
        fftshift 逐元素一致；
    (3) 原书 Listing 2.3 的高斯 DFT 与解析 FT 的最大偏差；
    (4) 原书 Listing 2.4 平移高斯的相位斜率恢复 -2*pi*x0（与解析相位比对），
        并复现"网格不够大时相位只在中心附近正确"的原书论断；
    (5) ift(ft(g)) 往返 + Parseval 定理。
MATLAB/Python 差异：
    - MATLAB 数组 1 起、numpy 0 起：居中排列的"零坐标"样本在 MATLAB 的
      N/2+1 号，numpy 的 N//2 号。坐标轴一律用 x = (n - N/2)*delta 生成，
      与排列解耦。
    - numpy.fft.fft 正变换不除 N（归一化在 ifft），MATLAB 相同——
      定标因子 delta / (N*delta_f) 都要自己乘。
用法：直接运行打印验证；`python ch2_ft_basics.py` 无图形输出。
依赖：numpy（不需要 matplotlib）。
"""
import os
import sys

import numpy as np

# ---- 复用《计算傅里叶光学》笔记的公共基础模块（同仓库） ----
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import gauss, coords  # noqa: E402


# ===== 书 2.1.3 节 Listing 2.1 / 2.2：定标正确的 DFT / DIFT =====
def ft(g, delta):
    """一维 DFT（原书 Listing 2.1）：
    G = fftshift(fft(fftshift(g))) * delta
    输入 g 按居中排列（零坐标样本在中间，偶数 N 时 numpy 索引 N//2），
    输出 G 同样居中排列；乘 delta 把求和升级为连续 FT 的近似。"""
    return np.fft.fftshift(np.fft.fft(np.fft.fftshift(g))) * delta


def ift(G, delta_f):
    """一维 DIFT（原书 Listing 2.2）：
    g = ifftshift(ifft(ifftshift(G))) * length(G) * delta_f
    逆变换的定标因子是 N*delta_f = 1/delta（保持 ft/ift 互逆）。"""
    return np.fft.ifftshift(np.fft.ifft(np.fft.ifftshift(G))) \
        * len(G) * delta_f


def main():
    print('===== 验证 1：式 (2.6) 的 DFT 就是重排后的黎曼和 =====')
    N, delta = 8, 0.5
    n = np.arange(N) - N // 2                      # n = -N/2 .. N/2-1
    x = n * delta
    g = gauss(x / 2.0)                             # 任意测试函数（居中排列）
    m = np.arange(N) - N // 2
    # 逐点按式 (2.4) 的指数计算（未重排的对称形式）
    G_sum = np.array([delta * np.sum(g * np.exp(-2j * np.pi * mi * n / N))
                      for mi in m])
    G_ft = ft(g, delta)
    print('    max|G_ft - G_黎曼和| = %.2e' % np.max(np.abs(G_ft - G_sum)))

    print('===== 验证 2：式 (2.5) 重排公式（原书笔误）=====')
    # 原书 (2.5) 第一支：g[(n'+N/2)*delta], n'=1..N/2+1 —— n'=1 时坐标
    # (1+N/2)*delta 超出采样范围 [-N/2, N/2-1]*delta，按字面无法取到样本；
    # 第二支 g[(n'-N-2)*delta] 整体错一个序号（末端只到 -2*delta）。
    # 正确的重排（与 fftshift 逐元素一致，也与原书图 2.1 一致）：
    #     g_tilde[n'] = g[(n'-1)*delta]        n' = 1 .. N/2   （1-based）
    #     g_tilde[n'] = g[(n'-1-N)*delta]      n' = N/2+1 .. N
    # 0-based numpy 写法：g_tilde = fftshift(g)（居中 -> 零坐标在最前）。
    book_idx = n[0]  # 仅占位避免未用警告
    del book_idx
    g_tilde_correct = np.concatenate([g[N // 2:], g[:N // 2]])
    g_tilde_fftshift = np.fft.fftshift(g)
    print('    修正公式 vs fftshift 逐元素最大差 = %.2e'
          % np.max(np.abs(g_tilde_correct - g_tilde_fftshift)))
    ok = ((n + N / 2) * delta <= x[-1] + 1e-12).mean()
    print('    原书第一支可取到的样本比例（N=%d）= %.0f%%（其余越界）'
          % (N, 100 * ok))

    print('===== 验证 3：Listing 2.3 高斯 DFT vs 解析 FT =====')
    # 原书参数：L = 5, N = 32, a = 1
    L, N, a = 5.0, 32, 1.0
    delta = L / N
    x, f = coords(L, N)          # x = n*delta（居中），f = n/(N*delta)
    g_samp = gauss(a * x)        # exp(-pi*(a*x)^2)
    G_dft = ft(g_samp, delta)
    G_ana = 1.0 / a * gauss(f / a)   # (1/|a|) exp(-pi*(f/a)^2)，式 (2.24)
    err = np.abs(G_dft - G_ana)
    i0 = np.argmin(np.abs(f))    # f = 0 的样本（偶数 N 时位于 N//2）
    print('    max|G_dft - G_ana| = %.2e，其中 f=0 处偏差 = %.2e'
          % (err.max(), err[i0]))

    print('===== 验证 4：Listing 2.4 平移高斯的相位 =====')
    # 原书参数：L = 10, N = 64, x0 = 5*delta, a = 1
    for (L, N) in [(5.0, 32), (10.0, 64)]:
        delta = L / N
        x0 = 5 * delta
        x, f = coords(L, N)
        g_samp = gauss(x - x0)                     # 平移后的高斯
        G_dft = ft(g_samp, delta)
        # 解析 FT：exp(-i 2 pi x0 f) * exp(-pi f^2)，用 -pi*f^2/a（a=1）
        G_ana = np.exp(-2j * np.pi * x0 * f) * gauss(f)
        # 只在幅度显著处（|G| > 1e-3 * 峰值）比较展开相位
        amp_ok = np.abs(G_dft) > 1e-3 * np.abs(G_dft).max()
        ph_err = np.angle(G_dft / G_ana)[amp_ok]
        print('    L=%-4.1f N=%d：幅度显著样本的相位误差 max = %.2e rad'
              % (L, N, np.max(np.abs(ph_err))))

    print('===== 验证 5：ft/ift 往返 + Parseval =====')
    L, N = 5.0, 64
    delta = L / N
    delta_f = 1.0 / (N * delta)
    x, f = coords(L, N)
    g = gauss(x / 1.5) * (1 + 0.3 * np.cos(2 * np.pi * 0.7 * x))
    g_rt = ift(ft(g, delta), delta_f)
    print('    max|ift(ft(g)) - g| = %.2e' % np.max(np.abs(g_rt - g)))
    lhs = np.sum(np.abs(g) ** 2) * delta
    rhs = np.sum(np.abs(ft(g, delta)) ** 2) * delta_f
    print('    Parseval: sum|g|^2*dx = %.10f, sum|G|^2*df = %.10f, 相对偏差 %.2e'
          % (lhs, rhs, abs(lhs - rhs) / lhs))


if __name__ == '__main__':
    main()


## 习题

`ch2_problems.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 2 章 =====
原书 2.7 节习题 1–5：五个信号在 a=1 与 a=10 两档下的 DFT vs 解析 FT

题目：对下列信号做 DFT，画出与解析 FT 的对比（a=1、a=10 各一次）：
    1) sinc(a x)              -> (1/a) rect(f/a)            [硬限带]
    2) exp(-pi (a x)^2)       -> (1/|a|) exp(-pi (f/a)^2)   [软限带]
    3) exp(-pi (a x)^2 + i pi (b x)^2), b=2
                              -> 1/sqrt(a^2-i b^2) exp(-pi f^2/(a^2-i b^2))
                                 [软限带，b 拉宽带宽]
    4) tri(a x)               -> (1/a) sinc^2(f/a)          [非限带，空域有限支撑]
    5) exp(-a |x|)            -> 2a/(a^2 + (2 pi f)^2)      [非限带，空域无限支撑]
采样策略（原书 2.5 节的案例研究就是这套方法的模板）：
    - sinc：奈奎斯特判据 delta = 0.85/a（留 15% 余量）；
    - 高斯类：p=0.01 电平带宽 -> delta = 1/(2 f_p)；
    - tri/exp(-a|x|)：谱按 1/f^2 衰减，取 |G| 降到峰值 p=0.01 的频率为
      f_max（tri 的 sinc^2 尾 -> f_max = a/(pi*sqrt(p))，洛伦兹尾 ->
      f_max = a sqrt(1/p-1)/(2 pi)），空域范围相应取振幅 < 0.01 处。
每个工况打印：delta、N、f_max 与 DFT 相对解析谱的最大偏差。
MATLAB 注意：tri 用 base 同款定义 |x|<=1 取 1-|x|；MATLAB 无内置 tri，
需自己写（见配套 .m 文件末尾的本地函数）。
用法：python ch2_problems.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib；base.py（同仓库 python/fourier/common）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import sinc, tri, coords  # noqa: E402


def ft1(g, delta):
    """居中一维 DFT（原书 Listing 2.1）"""
    return np.fft.fftshift(np.fft.fft(np.fft.fftshift(g))) * delta


# ---- 五个信号与解析 FT ----
def f_sinc(x, a):
    return sinc(a * x)


def G_sinc(f, a):
    return np.where(np.abs(f) < a / 2, 1.0 / a,
                    np.where(np.abs(np.asarray(f)) == a / 2, 0.5 / a, 0.0))


def f_gauss(x, a):
    return np.exp(-np.pi * (a * x) ** 2)


def G_gauss(f, a):
    return 1.0 / abs(a) * np.exp(-np.pi * (f / a) ** 2)


def f_gauss_quad(x, a, b=2.0):
    return np.exp(-np.pi * (a * x) ** 2) * np.exp(1j * np.pi * (b * x) ** 2)


def G_gauss_quad(f, a, b=2.0):
    c = a ** 2 - 1j * b ** 2
    return 1.0 / np.sqrt(c) * np.exp(-np.pi * f ** 2 / c)


def f_expdec(x, a):
    return np.exp(-a * np.abs(x))


def G_expdec(f, a):
    return 2 * a / (a ** 2 + (2 * np.pi * f) ** 2)


def bandwidth_p_general(G_func, a, p, f_hi=4000.0):
    """数值求 |G| 降到峰值 p 倍的频率（用于无闭式带宽的信号）"""
    f = np.linspace(1e-9, f_hi, 2_000_001)
    A = np.abs(G_func(f, a)) / np.abs(G_func(0.0, a))
    return np.interp(p, A[::-1], f[::-1])


def main(outdir=None):
    a_list = [1.0, 10.0]
    p = 0.01
    b3 = 2.0                              # 习题 3 指定 b = 2
    problems = [
        ('1: sinc(ax)',            lambda x, a: f_sinc(x, a),      G_sinc),
        ('2: exp(-pi a^2 x^2)',    lambda x, a: f_gauss(x, a),     G_gauss),
        ('3: quad-phase, b=2',     lambda x, a: f_gauss_quad(x, a, b3),
                                   lambda f, a: G_gauss_quad(f, a, b3)),
        ('4: tri(ax)',             lambda x, a: tri(a * x),
                                   lambda f, a: sinc(f / a) ** 2 / a),
        ('5: exp(-a|x|)',          lambda x, a: f_expdec(x, a),    G_expdec),
    ]

    print('===== 习题 1–5：a=1 / a=10 的采样参数与 DFT 精度 =====')
    print('%-18s %-5s %-10s %-8s %-12s %-12s' %
          ('信号', 'a', 'delta', 'N', '带内误差', '带外误差'))
    rows = []
    for name, fx, Gx in problems:
        for a in a_list:
            if name.startswith('1'):
                delta = 0.85 / a                    # 奈奎斯特 + 15% 余量
            elif name.startswith('4'):
                f_max = a / (np.pi * np.sqrt(p))    # sinc^2 尾：sinc^2(u)=p
                delta = 1.0 / (2 * f_max)
            elif name.startswith('5'):
                f_max = bandwidth_p_general(G_expdec, a, p)
                delta = 1.0 / (2 * f_max)
            else:
                # 信号 2：f_p = sqrt(a^2/pi*ln(1/p))；信号 3：式 (2.31)（b=2）
                coeff = a ** 2 if name.startswith('2') \
                    else a ** 2 + b3 ** 4 / a ** 2
                f_max = np.sqrt(coeff / np.pi * np.log(1 / p))
                delta = 1.0 / (2 * f_max)
            # 空域范围：截到信号幅度 < 0.01（tri 本身有限支撑，取 1.2/a）
            if name.startswith('4'):
                half = 1.2 / a
            elif name.startswith('1'):
                half = 12 / a                       # sinc 尾按 1/x 截到 ~3%
            elif name.startswith('5'):
                half = np.log(100) / a              # exp(-a|x|) 幅度 0.01 处
            else:
                half = np.sqrt(np.log(100) / np.pi) / a   # 高斯幅度 0.01 处
            N = int(2 ** np.ceil(np.log2(2 * half / delta)))   # 2 的幂
            x, f = coords(N * delta, N)
            g = fx(x, a)
            G = ft1(g, delta)
            G_a = Gx(f, a)
            peak = np.max(np.abs(G_a))
            dev = np.abs(np.abs(G) - np.abs(G_a)) / peak
            if name.startswith('1'):     # 硬限带：带内看波纹，带外看 Gibbs 溢出
                inb = np.abs(f) < a / 2
                err_in = np.max(dev[inb])
                err_out = np.max(dev[~inb])
            else:
                err_in = np.max(dev)
                err_out = float('nan')
            rows.append((name, a, delta, N, x, f, np.abs(G), np.abs(G_a),
                         err_in, err_out))
            print('%-18s %-5.0f %-10.4g %-8d %-12.2e %-12.4g'
                  % (name, a, delta, N, err_in, err_out))

    # ---- 绘图：5 行 x 2 列（a=1 左，a=10 右）；显示范围 = 0.8/delta ≈ 1.6 倍带宽 ----
    fig, axes = plt.subplots(5, 2, figsize=(12, 16))
    for r, (name, _, _) in enumerate(problems):
        for c, a in enumerate(a_list):
            row = next(q for q in rows if q[0] == name and q[1] == a)
            _, _, delta, N, x, f, Ga, Gan, err_in, err_out = row
            ax = axes[r, c]
            m = np.abs(f) <= 0.8 / delta
            ax.plot(f[m], Gan[m], color='0.6', lw=1.8, ls='--', label='解析')
            ax.plot(f[m], Ga[m], 'b.', ms=3, label='DFT')
            err_txt = err_in if err_out != err_out else \
                '%.1e/%.1e' % (err_in, err_out)
            ax.set_title('%s   a=%g   (delta=%.3g, N=%d, 误差 %s)'
                         % (name, a, delta, N, err_txt), fontsize=10)
            if r == 4:
                ax.set_xlabel(r'$f_x$ [m$^{-1}$]')
            if r == 0 and c == 0:
                ax.legend(fontsize=9)
            ax.grid(alpha=0.25)
    for ax in axes[:, 0]:
        ax.set_ylabel('幅值')
    fig.tight_layout()

    if outdir:
        os.makedirs(outdir, exist_ok=True)
        fp = os.path.join(outdir, 'problems.png')
        fig.savefig(fp, dpi=150)
        print('已保存：%s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)
